# Step 15: CNN, RNN, LSTM, GRU and Transformer
Every model reads the **25 frames** of each second. Same 3 folds as step 14.
*Same as `src/15_session_sequence.py`. Slow on a CPU (about an hour).*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page: https://multimediate-challenge.org/datasets/Dataset_NoXi/
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017, https://doi.org/10.1145/3136755.3136780
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

In [ ]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Load the 25-frame windows.** Columns 88-92 are the 5 bookkeeping columns, so they are removed.

In [ ]:
df = pd.read_parquet("data/features/noxi_engagement.parquet", columns=["engagement", "language", "session"])
y = df["engagement"].to_numpy("float32")
X = np.load("data/features/noxi_sequences.npy")
X = np.delete(X, [88, 89, 90, 91, 92], axis=2)
print(X.shape)      # windows x 25 frames x 797 features

In [ ]:
# the 3 sessions of each language
english = sorted(df[df["language"] == "English"]["session"].unique())
french = sorted(df[df["language"] == "French"]["session"].unique())
german = sorted(df[df["language"] == "German"]["session"].unique())

# fold 1 tests on the 1st session of each language, fold 2 on the 2nd, fold 3 on the 3rd
folds = []
for k in range(3):
    folds.append([english[k], french[k], german[k]])
print(folds)

**The same small head** at the end of every model: dropout, Dense(64), one output.

In [ ]:
def head(x):
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(64, activation="relu")(x)
    return layers.Dense(1)(x)

**CNN (VGG style):** two blocks of [convolution, convolution, pooling].

In [ ]:
def cnn():
    inp = layers.Input(shape=(25, 797))
    x = layers.Conv1D(32, 3, padding="same", activation="relu")(inp)
    x = layers.Conv1D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(2)(x)
    return keras.Model(inp, head(layers.Flatten()(x)), name="CNN_VGG")

**RNN, LSTM, GRU:** each reads the 25 frames one after another (128 units).

In [ ]:
def rnn():
    inp = layers.Input(shape=(25, 797))
    return keras.Model(inp, head(layers.SimpleRNN(128)(inp)), name="SimpleRNN")


def lstm():
    inp = layers.Input(shape=(25, 797))
    return keras.Model(inp, head(layers.LSTM(128)(inp)), name="LSTM")


def gru():
    inp = layers.Input(shape=(25, 797))
    return keras.Model(inp, head(layers.GRU(128)(inp)), name="GRU")

**Positional encoding:** attention does not know the order of the frames, so a learned vector is added for each of the 25 positions.

In [ ]:
class PositionalEncoding(layers.Layer):
    def build(self, input_shape):
        self.pos = self.add_weight(shape=(25, 64), initializer="random_normal", name="pos")

    def call(self, x):
        return x + self.pos

**Transformer:** frames → 64 numbers → add positions → 2 encoder blocks (self-attention + feed-forward, each with add & normalise) → average over frames → head.

In [ ]:
def encoder_block(x):
    attention = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.1)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, attention]))
    ff = layers.Dense(128, activation="relu")(x)
    ff = layers.Dropout(0.1)(ff)
    ff = layers.Dense(64)(ff)
    return layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, ff]))


def transformer():
    inp = layers.Input(shape=(25, 797))
    x = layers.Dense(64)(inp)
    x = PositionalEncoding()(x)
    x = encoder_block(x)
    x = encoder_block(x)
    x = layers.GlobalAveragePooling1D()(x)
    return keras.Model(inp, head(x), name="Transformer")

**Train every model on every fold.** Scaling learns only from the training frames; values limited to -5..5.

In [ ]:
scores = {"CNN_VGG": [], "SimpleRNN": [], "LSTM": [], "GRU": [], "Transformer": []}

for test_sessions in folds:
    test = df["session"].isin(test_sessions).to_numpy()
    train = ~test

    scaler = StandardScaler().fit(X[train].reshape(-1, 797))
    X_train = np.clip(scaler.transform(X[train].reshape(-1, 797)), -5, 5).reshape(-1, 25, 797).astype("float32")
    X_test = np.clip(scaler.transform(X[test].reshape(-1, 797)), -5, 5).reshape(-1, 25, 797).astype("float32")

    for make_model in [cnn, rnn, lstm, gru, transformer]:
        keras.utils.set_random_seed(42)
        model = make_model()
        model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
        model.fit(X_train, y[train], epochs=20, batch_size=128, verbose=0)
        scores[model.name].append(ccc(y[test], model.predict(X_test, verbose=0).flatten()))
    print("fold done:", test_sessions)

In [ ]:
saved = pd.read_csv("reports/session_sequence.csv").groupby("model")["ccc"].mean()
for name in scores:
    print(name.ljust(12), "mean", round(np.mean(scores[name]), 3), " saved", round(saved[name], 3))